# 2.8 FlashAttention V2：三个手术

论文：Dao, *FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning*（ICLR 2024）。

FA1 大胜之后，性能剖析发现它只用到 A100 理论算力的 25%~40%。FA2 的贡献不是新算法，而是三刀精准的手术：**算法微整、并行扩维、分工重构**。三刀过后，前向达到理论峰值的 50%~73%，整体比 FA1 快约 **2 倍**。

## 手术一：算法微整——rescale 后移、循环交换

（对照 2.7 节对比图的右半部分。）

1. **循环交换**：外层 KV → 外层 Q。每个 Q 块的状态 (m, l, acc) 只在自己的内层循环里存在，概念算法与工程实现从此统一；
2. **归一化后移**：O 未归一化累加，÷l 只在每个 Q 块收尾做一次；
3. **exp 基准直取全局**：P = exp(S − m_new)，省掉 FA1 的二次修正乘法。

还有两处随循环交换自然到位的细节：**O、l、m 的初始化直接在片上完成**（全 0 / −inf 的写操作落在 L1 / UB 上，不产生任何搬运——对比 FA1 中它们要从 HBM 读回）；收尾时每个 Q 块**一次性写出 O**，同时写出行统计量 **L = m + log(l)**（logsumexp）——把 m 与 l 合成一个数保存，供反向重计算（2.5 节第五节）使用，反向只需重算、无需回读前向的任何 n×n 矩阵。

效果（2.7 节实验已量化）：O 上的逐元素乘法少 1/3、除法次数降到 1/(块对数/Q块数)。数学结果逐位不变。

这些“省下的”运算在加速器上价值被放大：matmul 走 Tensor Core / Cube，逐元素只能走 CUDA core / Vector——**non-matmul 占比每降一点，整体速度就向峰值逼近一点**。

## 手术二：并行扩维——B × H × Q 块

![V2 的并行与 Warp 分工](images/fa2_gpu_optimization.svg)

FA1 只在 (B, H) 并行；FA2 增加序列维：**每个 Q 块（或若干 Q 块）一个独立任务**。用数字说话：

In [1]:
cores = 108                              # 以 A100 的 108 个 SM 为例
B, H, n, M = 1, 8, 32768, 128           # 长序列、小 batch 的典型场景
T_r = n // M                             # Q 块数 = 256

def util_bar(ratio, width=46):
    return '#' * int(round(ratio * width))

print(f'场景：B={B}, H={H}, n={n}, Q 块高 M={M} → Q 块数 T_r = {T_r}')
print(f'硬件：{cores} 个计算单元（以 A100 为例；昇腾的分核思路完全同构）\n')

for label, tasks in [('V1：任务数 = B×H         ', B * H),
                     ('V2：任务数 = B×H×T_r     ', B * H * T_r)]:
    ratio = min(tasks, cores) / cores
    print(f'{label}{tasks:5d} 个任务  占用率 {ratio:6.1%}  {util_bar(ratio)}')

场景：B=1, H=8, n=32768, Q 块高 M=128 → Q 块数 T_r = 256
硬件：108 个计算单元（以 A100 为例；昇腾的分核思路完全同构）

V1：任务数 = B×H              8 个任务  占用率   7.4%  ###
V2：任务数 = B×H×T_r       2048 个任务  占用率 100.0%  ##############################################

B=1、H=8、n=32k 的场景：FA1 只有 8 个任务（7% 占用），FA2 有 2048 个（满载）。两类现代负载尤其受益：

- **Decode**：每步只有 1 行 Q，靠把 (B × H × M 行 Q) 拼成足够多的任务才吃满硬件——GQA 模型 H 大，帮了大忙；
- **长文 Prefill**：n 大 → Q 块多，天然受益。

代价：不同的 Q 块要**各自遍历 KV**，KV 读取总量上升——好在多级缓存（L2 / 共享内存）能吸收。这也解释了为什么 Q 块不能切太碎：块高 M 太小，任务间 KV 重读开销就失控。

## 手术三：Warp 分工重构——按 Q 行切，零跨 Warp 通信

线程块内（如 128 行 Q、4 个 warp，见上图下半部分）：

| | FA1 | FA2 |
|--|--|--|
| 切分方式 | 按缩减维 K 切 4 份 | 按 Q 的行切 4 份 |
| 每 Warp 持有 | S 的部分列（32/128 维） | 32 行 Q 的完整状态 |
| rowmax / rowsum | **必须跨 Warp 归约**（共享内存 + 同步） | 各 Warp 私有，零通信 |
| K/V 访问 | 各 Warp 各读一份 | 广播共享 |

softmax 统计量 (m, l) 与 O 都**按 Q 的行私有**——按 Q 行切分后天然不需要任何跨 Warp 归约。FA1 的方式每步都要同步等待，是流水线上的硬气泡。

## 成绩单与昇腾映射

A100 上 FA2 前向达到理论峰值（~225 TFLOPS）的 50%~73%，整体较 FA1 约 2×；反向提升 1.5~1.75×。更重要的是——**它的三条设计原则在任何加速器上都成立**：

| FA2 的设计 | 昇腾 NPU 上的对应 | 本课程位置 |
|--|--|--|
| 外层 Q / 内层 KV | PromptFlashAttention kernel 的主循环结构 | 第 5 章 |
| B×H×Q块 并行 | AI Core 分核策略（GS1 等 Q 维切分） | 第 5、9 章 |
| 状态私有、少通信 | Cube/Vector 流水 + 双缓冲 | 第 1 章 2.3 / 2.6 节 |
| non-matmul 占比低 | Vector 单元负载（VF 建模） | 第 8 章 |
| Decode 并行不足的补救 | BN1 分核、负载均衡 | 第 5 章 |

> 换个角度读这张表：**FA2 的三个手术 = 第 5~9 章所有优化主题的纲**。GPU 上的 warp、Tensor Core 换成 NPU 上的 Cube / Vector / MTE，方法论原样成立。

## V2 之后：FA3 与稀疏化

- **FlashAttention-3**（2024，Hopper 架构）：三项硬件新特性的组合拳——
  **TMA**（张量内存加速器）：异步搬运单元，在「后台」把大块数据从全局内存搬进共享内存，计算与访存真正重叠；
  **WGMMA**（warp 组矩阵乘累加）：4 个 warp 协作执行一条异步 Tensor Core 指令，吞吐提升 2~3 倍，且可与其它单元并行；
  **FP8 低精度**：Tensor Core 吞吐直接翻倍（A100/H100 上 FP16 约 989 TFLOPS → FP8 约 1978 TFLOPS），用精度换速度；
  加上异步 warp specialization 让 Tensor Core 与 Softmax 真并行，利用率再上一个台阶；
- **稀疏化路线**：不再算全部 KV，只算“关键” token——DeepSeek V3.2 的 DSA、GLM-5 的稀疏注意力把计算量压到 1% 量级，昇腾上对应 SFA 算子。这条路线正是下一节的主角。

## 章节测验

1. FA2 的三个手术分别是什么？各自解决 FA1 的哪个“遗憾”？
2.（填空）FA2 的并行维度 = ____ × ____ × ____。
3. 为什么按 Q 行切 Warp 可以零通信，而按 K 维切必须跨 Warp 归约？（提示：softmax 统计量按什么维度私有）
4. 循环交换后 KV 读取总量上升，为什么实际仍能提速？FA2 靠什么参数控制这个代价？
5.（判断）FA2 修改了计算结果，换取了速度。（对/错）
6. FA2 的哪条设计在昇腾上映射为“Vector 单元负载优化”？对应第几章？
7. FA2 每个外层循环收尾写出 O 的同时还写出什么统计量？它由哪两个状态合成、供哪个技术使用？

> 答案见 `answer/02.08_answer.txt`。

下一节：[2.9 FA 的演进：MLA、GQA、SFA](02.09_fa_variants_mla_gqa_sfa.ipynb)——注意力家族的族谱，与昇腾 FA 算子的新战场。